# Aula 2 — Regressão Múltipla

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Se ainda não sabe como abrir e salvar sua própria cópia deste notebook,
veja a página **Antes de começar** no material da aula antes de continuar.

## Parte A: Demonstração

### Os dados: 300 apartamentos para alugar

Cada linha é um apartamento, com área, número de quartos, idade do
prédio, bairro e o aluguel cobrado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Endereço dos dados desta aula no GitHub.
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/alugueis.csv"
# Alternativa para testar offline, antes do repositório existir no GitHub:
# URL_DADOS = "../../data/alugueis.csv"

dados = pd.read_csv(URL_DADOS)
dados.head()

In [ ]:
# Quantos apartamentos existem em cada bairro, e qual o aluguel médio deles
print(dados["bairro"].value_counts())
print()
print(dados.groupby("bairro")["aluguel"].mean().round(2))

### Primeiro, só com a área

É o modelo da Aula 1, com uma variável só:

$$\hat{y} = w_0 + w_1 \cdot \text{área}$$

In [ ]:
from sklearn.linear_model import LinearRegression

modelo_area = LinearRegression()
modelo_area.fit(dados[["area_m2"]], dados["aluguel"])

r2_area = modelo_area.score(dados[["area_m2"]], dados["aluguel"])
print(f"Aluguel por m²: R$ {modelo_area.coef_[0]:.2f}")
print(f"R² usando só a área: {r2_area:.3f}")

### Agora com três variáveis

O modelo vira uma soma com uma parcela por informação:

$$\hat{y} = w_0 + \sum_{j=1}^{p} w_j x_j$$

Cada $w_j$ diz quanto o aluguel muda quando aquela informação sobe uma
unidade e todas as outras ficam paradas.

In [ ]:
colunas_numericas = ["area_m2", "quartos", "idade_anos"]

modelo_tres = LinearRegression()
modelo_tres.fit(dados[colunas_numericas], dados["aluguel"])

for nome, peso in zip(colunas_numericas, modelo_tres.coef_):
    print(f"{nome}: R$ {peso:.2f} por unidade")
print(f"R² com três variáveis: {modelo_tres.score(dados[colunas_numericas], dados['aluguel']):.3f}")

### O bairro vira colunas de 0 e 1

O bairro é um nome, não uma quantidade. Cada bairro vira uma coluna que
vale 1 quando o apartamento é dele e 0 quando não é. O `drop_first=True`
deixa um bairro de fora: ele é a categoria de referência, e o efeito dele
fica dentro do $w_0$.

In [ ]:
# Transforma a coluna de texto em colunas de 0 e 1 (variáveis indicadoras)
colunas_do_modelo = ["area_m2", "quartos", "idade_anos", "bairro"]
tabela_modelo = pd.get_dummies(dados[colunas_do_modelo], columns=["bairro"], drop_first=True)
tabela_modelo = tabela_modelo.astype(float)

print(tabela_modelo.head())
print()
print("Colunas do modelo:", list(tabela_modelo.columns))

In [ ]:
modelo_completo = LinearRegression()
modelo_completo.fit(tabela_modelo, dados["aluguel"])

print(f"w0 (valor base): R$ {modelo_completo.intercept_:.2f}")
for nome, peso in zip(tabela_modelo.columns, modelo_completo.coef_):
    print(f"{nome}: R$ {peso:.2f}")

### Prevendo um apartamento novo

Basta montar uma linha com as mesmas colunas, na mesma ordem.

In [ ]:
# Um apartamento de 70 m², 2 quartos, 10 anos, no Centro: as duas colunas
# de bairro valem 0, porque o Centro é a categoria de referência
apartamento_centro = pd.DataFrame({
    "area_m2": [70.0],
    "quartos": [2.0],
    "idade_anos": [10.0],
    "bairro_Jardins": [0.0],
    "bairro_Vila Nova": [0.0],
})
apartamento_jardins = apartamento_centro.copy()
apartamento_jardins["bairro_Jardins"] = 1.0

print(f"O mesmo apartamento no Centro:  R$ {modelo_completo.predict(apartamento_centro)[0]:.2f}")
print(f"O mesmo apartamento no Jardins: R$ {modelo_completo.predict(apartamento_jardins)[0]:.2f}")

### R² e R² ajustado

O R² nunca cai quando você adiciona uma variável. O R² ajustado desconta
o custo de cada variável nova:

$$R^2_{\text{aj}} = 1 - (1 - R^2) \cdot \frac{n - 1}{n - p - 1}$$

Aqui $n$ é o número de apartamentos e $p$ é o número de variáveis.

In [ ]:
def calcular_r2_ajustado(r2, n_linhas, n_variaveis):
    # Desconta do R2 o preco de cada variavel que entrou no modelo
    return 1 - (1 - r2) * (n_linhas - 1) / (n_linhas - n_variaveis - 1)

r2_completo = modelo_completo.score(tabela_modelo, dados["aluguel"])
r2_ajustado = calcular_r2_ajustado(r2_completo, len(dados), tabela_modelo.shape[1])

print(f"R² do modelo completo: {r2_completo:.3f}")
print(f"R² ajustado:           {r2_ajustado:.3f}")

In [ ]:
# O gráfico de diagnóstico: resíduo contra valor previsto
alugueis_previstos = modelo_completo.predict(tabela_modelo)
residuos = dados["aluguel"] - alugueis_previstos

plt.scatter(alugueis_previstos, residuos)
plt.axhline(0)
plt.xlabel("Aluguel previsto (R$)")
plt.ylabel("Resíduo (R$)")
plt.title("Resíduos contra previsto")
plt.show()

### Separando treino e teste

Até aqui medimos o modelo nos mesmos dados em que ele treinou. Isso é
otimista. O jeito honesto é guardar apartamentos que ele nunca viu.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

treino_x, teste_x, treino_y, teste_y = train_test_split(
    tabela_modelo, dados["aluguel"], test_size=0.3, random_state=42)

print(f"treino: {len(treino_x)} apartamentos")
print(f"teste:  {len(teste_x)} apartamentos")

modelo_split = LinearRegression().fit(treino_x, treino_y)
print(f"R² no treino: {r2_score(treino_y, modelo_split.predict(treino_x)):.3f}")
print(f"R² no teste:  {r2_score(teste_y, modelo_split.predict(teste_x)):.3f}")

### O sobreajuste, visto de perto

Vamos encher o modelo de colunas de números sorteados. Elas não sabem
nada sobre aluguel. Repare no que acontece com cada R².

In [ ]:
def montar_com_ruido(quantas_colunas):
    # As 5 colunas de verdade, mais colunas de numeros sorteados
    tabela = tabela_modelo.copy()
    sorteio = np.random.default_rng(42)
    for j in range(quantas_colunas):
        tabela[f"ruido_{j + 1}"] = sorteio.normal(size=len(dados))
    return tabela

print(f"{'colunas de ruído':>18} {'R² treino':>10} {'R² teste':>10} {'MAE teste':>12}")
for quantas in [0, 10, 20, 40]:
    tabela = montar_com_ruido(quantas)
    tr_x, te_x, tr_y, te_y = train_test_split(
        tabela, dados["aluguel"], test_size=0.3, random_state=42)
    m = LinearRegression().fit(tr_x, tr_y)
    print(f"{quantas:>18} {r2_score(tr_y, m.predict(tr_x)):>10.3f} "
          f"{r2_score(te_y, m.predict(te_x)):>10.3f} "
          f"{mean_absolute_error(te_y, m.predict(te_x)):>11.0f}")

print()
print("O R² de treino sobe e o de teste desce. Isso se chama sobreajuste.")

### Padronizar: obrigatório antes de regularizar

A regularização olha o tamanho dos coeficientes, e o tamanho depende da
unidade da variável. Área em m² e idade em anos não são comparáveis.

$$x_{\text{padronizado}} = \frac{x - \text{média}}{\text{desvio padrão}}$$

A média e o desvio saem **só do treino**. O teste usa a mesma escala.

In [ ]:
from sklearn.preprocessing import StandardScaler

tabela_ruido = montar_com_ruido(40)
treino_x, teste_x, treino_y, teste_y = train_test_split(
    tabela_ruido, dados["aluguel"], test_size=0.3, random_state=42)

escala = StandardScaler().fit(treino_x)
treino_p = escala.transform(treino_x)
teste_p = escala.transform(teste_x)

print(f"{treino_p.shape[1]} colunas: 5 de verdade e 40 sorteadas")
print(f"média de cada coluna depois de padronizar: {treino_p.mean():.2f}")
print(f"desvio de cada coluna depois de padronizar: {treino_p.std():.2f}")

### As três regularizações

Todas somam ao erro uma penalidade pelo tamanho dos coeficientes:

$$\text{custo} = \text{MSE} + \alpha \cdot \text{penalidade}(w)$$

O que muda é a conta da penalidade:

$$\text{Ridge}: \sum_j w_j^2 \qquad \text{Lasso}: \sum_j |w_j| \qquad
\text{ElasticNet}: \rho \sum_j |w_j| + \frac{1-\rho}{2} \sum_j w_j^2$$

As versões `CV` escolhem o $\alpha$ sozinhas, por validação cruzada.

In [ ]:
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV

forcas = np.logspace(-2, 3, 60)

modelos = {
    "Sem freio": LinearRegression(),
    "Ridge": RidgeCV(alphas=forcas),
    "Lasso": LassoCV(alphas=forcas, cv=5, max_iter=50000, random_state=0),
    "ElasticNet": ElasticNetCV(alphas=forcas, l1_ratio=[0.1, 0.5, 0.9, 1.0],
                               cv=5, max_iter=50000, random_state=0),
}

print(f"{'modelo':<12} {'alpha':>8} {'R² teste':>9} {'MAE teste':>10} {'zerados':>9}")
for nome, modelo in modelos.items():
    modelo.fit(treino_p, treino_y)
    previsto = modelo.predict(teste_p)
    alpha = f"{modelo.alpha_:.2f}" if hasattr(modelo, "alpha_") else "-"
    zerados = int((np.abs(modelo.coef_) < 1e-8).sum())
    print(f"{nome:<12} {alpha:>8} {r2_score(teste_y, previsto):>9.3f} "
          f"{mean_absolute_error(teste_y, previsto):>9.0f} {zerados:>6}/45")

print()
print(f"O ElasticNet escolheu l1_ratio = {modelos['ElasticNet'].l1_ratio_}")
print("Com l1_ratio = 1,0 ele é o Lasso puro: perguntamos qual ferramenta")
print("usar, e o modelo respondeu.")

In [ ]:
# Onde cada modelo colocou o peso: nas 5 colunas de verdade ou no ruido?
print(f"{'modelo':<12} {'peso nas 5 reais':>18} {'peso nas 40 de ruído':>22}")
for nome, modelo in modelos.items():
    reais = np.abs(modelo.coef_[:5]).sum()
    ruido = np.abs(modelo.coef_[5:]).sum()
    print(f"{nome:<12} {reais:>18.0f} {ruido:>22.0f}")

print()
print("O Ridge encolhe o ruído de pouquinho. O Lasso corta.")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo os apartamentos

Rode a célula abaixo e observe: qual é a área do maior apartamento? E a
idade do prédio mais antigo?

In [ ]:
dados.describe()

In [ ]:
if len(dados) == 300:
    print(f"✅ Os dados têm {len(dados)} apartamentos, como esperado.")
else:
    print("❌ Confira se você rodou a célula que carrega os dados, no início do notebook.")

### Exercício 2: o aluguel médio de cada bairro

Rode a célula e responda: o bairro com maior área média é também o mais
caro?

In [ ]:
resumo_bairros = dados.groupby("bairro")[["area_m2", "aluguel"]].mean().round(1)
print(resumo_bairros)

In [ ]:
print("Converse com um colega: a Vila Nova tem a maior área média e o menor aluguel médio. Por quê?")

### Exercício 3: criando as colunas de bairro

Complete a linha que transforma a coluna `bairro` em colunas de 0 e 1.
Use `pd.get_dummies` com `columns=["bairro"]` e `drop_first=True`.

In [ ]:
colunas_escolhidas = ["area_m2", "quartos", "idade_anos", "bairro"]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(minha_tabela.head())

In [ ]:
if minha_tabela.shape[1] == 5:
    print("✅ Cinco colunas: três numéricas e duas de bairro. É isso mesmo.")
else:
    print("❌ Esperava 5 colunas. Você usou drop_first=True para deixar um bairro de fora?")

### Exercício 4: treinando o modelo completo

Treine um `LinearRegression` chamado `meu_modelo` usando `minha_tabela`
como entrada e a coluna `aluguel` como alvo.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"w0: R$ {meu_modelo.intercept_:.2f}")
for nome, peso in zip(minha_tabela.columns, meu_modelo.coef_):
    print(f"{nome}: R$ {peso:.2f}")

In [ ]:
if abs(meu_modelo.coef_[0] - 25.1) < 1:
    print("✅ O coeficiente da área ficou perto de R$ 25 por m², como esperado.")
else:
    print("❌ Confira se minha_tabela tem as cinco colunas e se o alvo é dados['aluguel'].")

### Exercício 5: R² e R² ajustado

Calcule o R² do seu modelo e depois o R² ajustado, usando a função
`calcular_r2_ajustado` que já existe no notebook:

$$R^2_{\text{aj}} = 1 - (1 - R^2) \cdot \frac{n - 1}{n - p - 1}$$

In [ ]:
n_linhas = len(dados)
n_variaveis = minha_tabela.shape[1]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"R²: {meu_r2:.3f}")
print(f"R² ajustado: {meu_r2_ajustado:.3f}")

In [ ]:
if meu_r2 > 0.9 and meu_r2_ajustado < meu_r2:
    print("✅ R² alto, e o ajustado um pouquinho menor. É exatamente o esperado.")
else:
    print("❌ Confira a ordem dos argumentos: primeiro o R², depois n e por último p.")

### Exercício 6: prevendo um apartamento

Monte uma linha para um apartamento de 90 m², 3 quartos, 5 anos de
prédio, na **Vila Nova**, e preveja o aluguel. Lembre: `bairro_Jardins`
vale 0 e `bairro_Vila Nova` vale 1.

In [ ]:
# As colunas precisam ser as mesmas, na mesma ordem do treino
print("Ordem das colunas:", list(minha_tabela.columns))

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Aluguel previsto: R$ {aluguel_previsto:.2f}")

In [ ]:
if 2000 < aluguel_previsto < 3500:
    print("✅ O valor está na faixa esperada para esse apartamento.")
else:
    print("❌ Confira os valores das colunas de bairro: Jardins = 0 e Vila Nova = 1.")

### Exercício 7: o coeficiente que muda

Treine um modelo só com a área e compare o coeficiente dela com o do
modelo completo. Guarde os dois valores para responder à pergunta abaixo.

In [ ]:
modelo_so_area = LinearRegression()
modelo_so_area.fit(dados[["area_m2"]], dados["aluguel"])

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Coeficiente da área sozinha:  R$ {coef_area_sozinha:.2f} por m²")
print(f"Coeficiente da área no modelo completo: R$ {coef_area_completo:.2f} por m²")

In [ ]:
if coef_area_sozinha > coef_area_completo:
    print("✅ O coeficiente encolheu quando as outras variáveis entraram.")
    print("   A área dividiu o crédito com os quartos, que andam junto com ela.")
else:
    print("❌ Confira se você comparou os dois modelos certos.")

### Exercício 8: separando treino e teste

Separe 30% dos apartamentos para teste, com `random_state=42`, e compare
o R² nos dois conjuntos. Use `minha_tabela_ruido`, que já tem as 40
colunas sorteadas.

In [ ]:
minha_tabela_ruido = montar_com_ruido(40)
print(f"{minha_tabela_ruido.shape[1]} colunas no total")

In [ ]:
# SEU CODIGO AQUI

In [ ]:
modelo_sem_freio = LinearRegression().fit(meu_treino_x, meu_treino_y)
r2_treino = r2_score(meu_treino_y, modelo_sem_freio.predict(meu_treino_x))
r2_teste = r2_score(meu_teste_y, modelo_sem_freio.predict(meu_teste_x))

print(f"R² no treino: {r2_treino:.3f}")
print(f"R² no teste:  {r2_teste:.3f}")

In [ ]:
if len(meu_teste_x) == 90 and r2_treino > r2_teste:
    print("✅ 90 apartamentos guardados, e o R² de treino é maior que o de teste.")
    print("   Essa distância é o sobreajuste que a regularização vai atacar.")
else:
    print("❌ Confira o test_size=0.3 e o random_state=42.")

### Exercício 9: padronizando

Crie `minha_escala` com `StandardScaler`, ajustada **só no treino**, e
aplique nos dois conjuntos.

$$x_{\text{padronizado}} = \frac{x - \text{média}}{\text{desvio padrão}}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"média das colunas no treino padronizado: {meu_treino_p.mean():.3f}")
print(f"desvio das colunas no treino padronizado: {meu_treino_p.std():.3f}")
print(f"média das colunas no teste padronizado:  {meu_teste_p.mean():.3f}")

In [ ]:
if abs(meu_treino_p.mean()) < 0.01 and abs(meu_treino_p.std() - 1) < 0.01:
    print("✅ Treino com média 0 e desvio 1.")
    print("   Repare que o teste não fica exatamente em 0: é o esperado,")
    print("   porque ele usa a média e o desvio do treino, e não os dele.")
else:
    print("❌ Confira: fit só no treino, e transform nos dois.")

### Exercício 10: as três regularizações

Treine `RidgeCV`, `LassoCV` e `ElasticNetCV` nos dados padronizados e
compare com a regressão sem freio. Deixe o `ElasticNetCV` escolher o
`l1_ratio` entre `[0.1, 0.5, 0.9, 1.0]`.

In [ ]:
minhas_forcas = np.logspace(-2, 3, 60)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
meu_sem_freio = LinearRegression().fit(meu_treino_p, meu_treino_y)
for nome, modelo in [("Sem freio", meu_sem_freio), ("Ridge", meu_ridge),
                     ("Lasso", meu_lasso), ("ElasticNet", meu_elastic)]:
    erro = mean_absolute_error(meu_teste_y, modelo.predict(meu_teste_p))
    zerados = int((np.abs(modelo.coef_) < 1e-8).sum())
    print(f"{nome:<12} MAE no teste R$ {erro:>4.0f}   coeficientes zerados: {zerados:>2}/45")

In [ ]:
zerados_lasso = int((np.abs(meu_lasso.coef_) < 1e-8).sum())
erro_lasso = mean_absolute_error(meu_teste_y, meu_lasso.predict(meu_teste_p))
erro_sem = mean_absolute_error(meu_teste_y, meu_sem_freio.predict(meu_teste_p))
if zerados_lasso > 15 and erro_lasso < erro_sem:
    print(f"✅ O Lasso zerou {zerados_lasso} colunas e errou R$ {erro_sem - erro_lasso:.0f} menos.")
    print(f"   O ElasticNet escolheu l1_ratio = {meu_elastic.l1_ratio_}.")
    print("   Com 1,0 ele virou Lasso puro: o próprio modelo escolheu a ferramenta.")
else:
    print("❌ Confira se você treinou nos dados padronizados (meu_treino_p).")

### Exercício 11: desafio, o erro do dia

Rode o mesmo `Lasso` duas vezes, uma **sem** padronizar e outra
padronizada, e compare os coeficientes das 5 colunas de verdade.

Dica: use `Lasso(alpha=50, max_iter=50000)` nos dois casos.

In [ ]:
from sklearn.linear_model import Lasso

nomes_reais = list(minha_tabela_ruido.columns[:5])

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"{'coluna':<18} {'sem padronizar':>16} {'padronizado':>14}")
for i, nome in enumerate(nomes_reais):
    print(f"{nome:<18} {lasso_cru.coef_[i]:>16.2f} {lasso_padronizado.coef_[i]:>14.2f}")

In [ ]:
maior_cru = nomes_reais[int(np.argmax(np.abs(lasso_cru.coef_[:5])))]
maior_pad = nomes_reais[int(np.argmax(np.abs(lasso_padronizado.coef_[:5])))]
print()
print(f"Sem padronizar, a coluna que parece mais forte é: {maior_cru}")
print(f"Padronizado, a coluna que de fato é mais forte é: {maior_pad}")
if maior_cru != maior_pad:
    print("✅ A ordem se inverteu. Sem padronizar, o freio julga pela unidade")
    print("   da variável (m², anos, 0 ou 1) e não pela importância dela.")
else:
    print("Compare os números: as escalas mudam muito entre as duas colunas.")

Agora, em texto: escreva 3 a 4 frases explicando (a) por que o
coeficiente da área diminui quando os quartos entram no modelo e (b) por
que o Lasso zera coeficientes e o Ridge não. Edite esta célula (duplo
clique nela) e escreva sua resposta no lugar deste parágrafo.